In [ ]:
from pathlib import Path

from experiments import trainers as tr
from experiments.load_data import load_data

In [ ]:
DEFAULT_DATA_DIR = Path("/Volumes") / "Crucial X9" # local machine

user_list = sorted([
        p for p in Path(DEFAULT_DATA_DIR, "emg_dataset/by_user").iterdir()
        if p.is_dir()
    ])

In [ ]:
user_train_dict, held_out_session = load_data("single_user", user_list)
user_train_dict

In [ ]:
# Inspect Meta's Pretrained Model

from emg2pose.utils import generate_hydra_config_from_overrides
from emg2pose.lightning import Emg2PoseModule

meta_ckpt = Path(DEFAULT_DATA_DIR) / "emg2pose_model_checkpoints" / "regression_vemg2pose.ckpt"

config = generate_hydra_config_from_overrides(
    overrides=[
        "experiment=tracking_vemg2pose",
        f"checkpoint={DEFAULT_DATA_DIR}/emg2pose_model_checkpoints/regression_vemg2pose.ckpt"
    ]
)

module = Emg2PoseModule.load_from_checkpoint(
    config.checkpoint,
    network=config.network,
    optimizer=config.optimizer,
    lr_scheduler=config.lr_scheduler,
)

module

In [ ]:
# inspect parameters

for name, param in module.named_parameters():
    print(name)

In [ ]:
# freeze mlp layers

for name, param in module.named_parameters():
    param.requires_grad = False

    if "model.decoder.mlp_out" in name:
        param.requires_grad = True

# check what is unfrozen
for name, param in module.named_parameters():
    if param.requires_grad:
        print(name)

In [ ]:
fine_tuned_model = tr.fine_tune_emg2pose(user_train_dict, DEFAULT_DATA_DIR, epochs=5)

In [ ]:
from emg2pose.data import Emg2PoseSessionData

data = Emg2PoseSessionData(hdf5_path=held_out_session)

In [ ]:
from experiments.inference import emg2pose_inferece
from run_experiment import timer
from experiments.metrics import ExperimentMetrics

metrics_rows = []

# Meta EMG2Pose Fine Tuned
with timer("Meta emg2pose"):
    preds, joint_angles, no_ik_failure = emg2pose_inferece(data, model)
    m = ExperimentMetrics(preds, joint_angles, no_ik_failure)
    metrics_rows.append({
        "Model": "Meta Fine-Tuned",
        **m.main
    })

m.main


In [ ]:
meta_model = tr.get_emg2pose(DEFAULT_DATA_DIR)

In [ ]:
from experiments.inference import emg2pose_inferece
from run_experiment import timer
from experiments.metrics import ExperimentMetrics

metrics_rows = []

# Meta EMG2Pose Fine Tuned
with timer("Meta emg2pose"):
    preds, joint_angles, no_ik_failure = emg2pose_inferece(data, meta_model)
    m = ExperimentMetrics(preds, joint_angles, no_ik_failure)
    metrics_rows.append({
        "Model": "Meta Fine-Tuned",
        **m.main
    })

m.main
